In [1]:
!pip install -U unsloth transformers==4.56.2 trl==0.22.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 90.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 29.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 77.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 12.2 MB/s eta 0:00

In [ ]:
%%writefile pretokenize.py
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_dataset
import os


MODEL_NAME = "CohereLabs/tiny-aya-base"
MAX_SEQ_LENGTH = 1024
CONDITION_NAME = "condition-1-en-5k"

TRAIN_OUT = f"/kaggle/working/data/{CONDITION_NAME}/train_tokenized"


def main():
    os.makedirs(os.path.dirname(TRAIN_OUT), exist_ok=True)

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("hf")

    # We only need the tokenizer here.
    _, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
        token=hf_token,
    )

    train_dataset = load_dataset(
        "legesher/language-decoded-data",
        CONDITION_NAME,
        split="train",
    )

    def tokenize_fn(batch):
        return tokenizer(
            batch["code"],
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=False,
        )

    # Keep only tokenized fields to reduce disk size.
    train_tokenized = train_dataset.map(
        tokenize_fn,
        batched=True,
        num_proc=8,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train dataset",
    )

    train_tokenized.save_to_disk(TRAIN_OUT)

    print(f"Saved train tokenized dataset to: {TRAIN_OUT}")
    print(train_tokenized)


if __name__ == "__main__":
    main()

Writing pretokenize.py


In [ ]:
%%writefile train.py
# train.py
from kaggle_secrets import UserSecretsClient
from unsloth import FastModel
from datasets import load_from_disk
from transformers import DataCollatorForLanguageModeling
from trl import SFTTrainer, SFTConfig
import torch
import os


MODEL_NAME = "CohereLabs/tiny-aya-base"
MAX_SEQ_LENGTH = 1024
CONDITION_NAME =  "condition-1-en-5k"

TRAIN_PATH = f"/kaggle/working/data/{CONDITION_NAME}/train_tokenized"
OUTPUT_DIR = f"/kaggle/working/output/{CONDITION_NAME}"


def is_main_process():
    return int(os.environ.get("RANK", "0")) == 0


def main():
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("hf")

    # Helpful for near-OOM situations.
    os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

    model, tokenizer = FastModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        full_finetuning=False,
        token=hf_token,
        # Do not set device_map="auto" under DDP
    )

    model = FastModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        random_state=42,
        gradient_checkpointing="unsloth",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
    )

    train_dataset = load_from_disk(TRAIN_PATH)

    # Since the dataset is already tokenized, use a collator directly.
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
    )

    if is_main_process():
        gpu_stats = torch.cuda.get_device_properties(0)
        start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        max_memory = round(gpu_stats.total_memory / 1024**3, 3)
        print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
        print(f"{start_gpu_memory} GB of memory reserved.")
        print(train_dataset)

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        data_collator=data_collator,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,

            # Since tokenization is already done, do not specify dataset_text_field.
            max_seq_length=MAX_SEQ_LENGTH,

            max_grad_norm=1.0,
            per_device_train_batch_size=8,
            gradient_accumulation_steps=1,

            warmup_ratio=0.05,
            num_train_epochs = 1, # Set this for 1 full training run.
            learning_rate=2e-4,
            packing=True,  
            logging_steps=10,
            optim="paged_adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="cosine",
            report_to="none",
            fp16=True,
            bf16=False,
            eval_strategy="no",
            save_strategy="steps",
            save_steps=500,
            save_total_limit=2,

            run_name=CONDITION_NAME,
            ddp_find_unused_parameters=False,

            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            dataloader_persistent_workers=True,
        ),
    )

    trainer_stats = trainer.train()

    if is_main_process():
        used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
        print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
        print(f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training.")
        print(f"Peak reserved memory = {used_memory} GB.")

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        model.push_to_hub("legesher/language-decoded-lora", token=hf_token, subfolder=CONDITION_NAME)
        print(f"Saved final model to: {OUTPUT_DIR}")
        import json
        with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
            json.dump(trainer.state.log_history, f, indent=2)
        
        from huggingface_hub import HfApi
        api = HfApi()

        api.upload_file(
            path_or_fileobj=os.path.join(OUTPUT_DIR, "training_metrics.json"),
            path_in_repo=f"{CONDITION_NAME}/training_metrics.json",
            repo_id="legesher/language-decoded-lora",
            token=hf_token,
        )

        print("Uploaded training_metrics.json to Hugging Face Hub.")


if __name__ == "__main__":
    main()

Writing train.py


In [ ]:
!python pretokenize.py

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-22 07:20:57.030243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774164057.426888     131 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774164057.525743     131 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774164058.543992     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774164058.544040     131 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:17

In [17]:
!torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py

W0322 06:03:01.654000 3574 torch/distributed/run.py:852] 
W0322 06:03:01.654000 3574 torch/distributed/run.py:852] *****************************************
W0322 06:03:01.654000 3574 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0322 06:03:01.654000 3574 torch/distributed/run.py:852] *****************************************
[W322 06:03:02.411596942 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-03-22 06:03:06.443647: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-22 